# Project 2: Git Clone & Environment Setup

In [1]:
# 1. Clone or Pull Repository
%cd /content/

import os
REPO_URL = "https://github.com/cyunchaeskku/AILab_Project2.git"
REPO_NAME = "AILab_Project2"

if os.path.exists(REPO_NAME):
    print(f"Pulling latest changes for {REPO_NAME}...")
    !cd {REPO_NAME} && git pull
else:
    print(f"Cloning {REPO_NAME}...")
    !git clone {REPO_URL}

/content
Pulling latest changes for AILab_Project2...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 10.37 KiB | 3.46 MiB/s, done.
From https://github.com/cyunchaeskku/AILab_Project2
   bc5e7e1..ae6148d  main       -> origin/main
Updating bc5e7e1..ae6148d
Fast-forward
 configs/progressive_1024_a100.yaml |  18 +-
 configs/progressive_1024_l4.yaml   |  18 +-
 test.ipynb                         | 423 +++++++++++++++++++++++++++++++++++--
 3 files changed, 427 insertions(+), 32 deletions(-)


In [9]:
%cd AILab_Project2

In [2]:
# 2. Install dependencies
!pip install gdown
%cd AILab_Project2
!pip install -r requirements.txt

/content/AILab_Project2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 131.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 126.0 MB/s eta 0:00:0000:010:01


In [3]:
# 3. Mount Google Drive for Checkpoints
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_PROJECT = Path("/content/drive/MyDrive/OpenAILab/project2")
(DRIVE_PROJECT / "512").mkdir(parents=True, exist_ok=True)
(DRIVE_PROJECT / "1024").mkdir(parents=True, exist_ok=True)
!ls -lh /content/drive/MyDrive/OpenAILab/project2


Mounted at /content/drive
total 28K
drwx------ 2 root root 4.0K May 28 07:46 1024
drwx------ 2 root root 4.0K May 27 15:44 512
drwx------ 2 root root 4.0K May 27 15:35 checkpoints
drwx------ 2 root root 4.0K May 28 14:00 pggan_1024
drwx------ 2 root root 4.0K May 31 08:55 pggan_1024_fp32_safer
drwx------ 2 root root 4.0K May 30 12:38 pggan_1024_fp32_stable
drwx------ 2 root root 4.0K May 31 02:53 train_val_data


In [4]:
# 4. Download Train/Validation Datasets
from pathlib import Path
import shutil
import subprocess

DATA_DIR = Path("data")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/OpenAILab/project2/train_val_data")
DATA_DIR.mkdir(exist_ok=True)
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

files = {
    # Training data only. Use these for model training.
    "train_50k_256.zip": "1Bx2wH_ZJ44RXEY5X5pJJyYY8NaZbHa9w",
    # "train_50k_512.zip": "1Sckvcw27OggN49FeOTN6_-xg-VZVNr8I",
    "train_50k_512.zip": "1R1JBbYGfeOPZedKHjuJ5VZavVdjXuJl-",
    # "train_50k_1024.zip": "1PhuCuv9cRNYUj1ItiSUUaX0_2zOBQjQs",
    "train_50k_1024.zip": "1qFJN3QOmM8kjwgWKztHPIqTvZFuZnZLd",

    # Validation data only. Do not use these for training.
    "valid_10k_256.zip": "1yEA26_FEgHwPur2jZCr0lmONesA6TyI2",
    "valid_10k_512.zip": "1hLDAyFgTZA2KRYSdJvPLhHJ3K0AgWfpE",
    "valid_10k_1024.zip": "1V0HfqDxUytt-wOiqyerXYC1-61EE8HqZ",
}

for filename, file_id in files.items():
    out = DATA_DIR / filename
    cached = DRIVE_DATA_DIR / filename

    if out.exists() and out.stat().st_size > 0:
        print(f"skip local {out} ({out.stat().st_size / 1024**3:.2f} GB)")
        if not cached.exists() or cached.stat().st_size == 0:
            print(f"cache to Drive {cached}")
            shutil.copy2(out, cached)
        continue

    if cached.exists() and cached.stat().st_size > 0:
        print(f"copy from Drive cache {cached} -> {out}")
        shutil.copy2(cached, out)
        continue

    print(f"download {out}")
    url = f"https://drive.google.com/uc?id={file_id}"
    result = subprocess.run(["gdown", url, "-O", str(out)], check=False)
    if result.returncode != 0 or not out.exists() or out.stat().st_size == 0:
        print(f"download failed: {filename}")
        if out.exists() and out.stat().st_size == 0:
            out.unlink()
        continue

    print(f"cache to Drive {cached}")
    shutil.copy2(out, cached)

!ls -lh data
!ls -lh /content/drive/MyDrive/OpenAILab/project2/train_val_data


copy from Drive cache /content/drive/MyDrive/OpenAILab/project2/train_val_data/train_50k_256.zip -> data/train_50k_256.zip
copy from Drive cache /content/drive/MyDrive/OpenAILab/project2/train_val_data/train_50k_512.zip -> data/train_50k_512.zip
copy from Drive cache /content/drive/MyDrive/OpenAILab/project2/train_val_data/train_50k_1024.zip -> data/train_50k_1024.zip
copy from Drive cache /content/drive/MyDrive/OpenAILab/project2/train_val_data/valid_10k_256.zip -> data/valid_10k_256.zip
copy from Drive cache /content/drive/MyDrive/OpenAILab/project2/train_val_data/valid_10k_512.zip -> data/valid_10k_512.zip
copy from Drive cache /content/drive/MyDrive/OpenAILab/project2/train_val_data/valid_10k_1024.zip -> data/valid_10k_1024.zip
total 29G
-rw------- 1 root root  17G May 31 03:08 train_50k_1024.zip
-rw------- 1 root root 1.6G May 15 03:47 train_50k_256.zip
-rw------- 1 root root 5.2G May 31 03:06 train_50k_512.zip
-rw------- 1 root root 3.4G May 15 04:01 valid_10k_1024.zip
-rw-------

In [ ]:
# 5. Verify Baseline Sample
!python generate.py --ckpt ckpt/ffhq256_baseline.pt --out sample_256.png --n 16 --seed 12345

In [5]:
# 6. Login to Weights & Biases
import getpass
import os
import wandb

if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = getpass.getpass("WANDB_API_KEY: ")

wandb.login(key=os.environ["WANDB_API_KEY"])


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cyunchaeskku (cyunchaeskku-sungkyunkwan-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
# 7. Clean GPU Processes
import gc
import os
import signal
import subprocess

print("GPU processes before cleanup:")
!nvidia-smi

current_pid = os.getpid()
cmd = [
    "nvidia-smi",
    "--query-compute-apps=pid,process_name,used_memory",
    "--format=csv,noheader,nounits",
]
result = subprocess.run(cmd, capture_output=True, text=True)

for line in result.stdout.strip().splitlines():
    if not line.strip():
        continue
    pid_text, name, mem_text = [part.strip() for part in line.split(",", 2)]
    pid = int(pid_text)
    if pid == current_pid:
        print(f"keep current kernel pid={pid} mem={mem_text}MiB")
        continue
    print(f"kill gpu process pid={pid} name={name} mem={mem_text}MiB")
    os.kill(pid, signal.SIGTERM)

gc.collect()
try:
    import torch
    torch.cuda.empty|_cache()
except Exception as exc:
    print(f"torch cache cleanup skipped: {exc}")

print("GPU processes after cleanup:")
!nvidia-smi


In [ ]:
# 8. Start True Progressive Training from Professor 256 Baseline
# Uses PG-GAN-style fade-in schedule: 256 stabilize -> 512 fade/stabilize -> 1024 fade/stabilize.
!python train.py --config configs/progressive_1024.yaml --init-from ckpt/ffhq256_baseline.pt
# !python train.py --config configs/progressive_1024.yaml --init-from ckpt/ffhq256_baseline.pt --auto-resume
# !timeout --signal=INT --kill-after=60s 5h python train.py --config configs/progressive_1024.yaml --auto-resume


Auto config selected: configs/progressive_1024_a100.yaml (gpu=NVIDIA A100-SXM4-40GB)
Auto-resume found no checkpoint; starting from --init-from.
Generator: 21.31M params
Discriminator: 20.25M params
Optimizers: G lr=0.0005, D lr=0.0005
Progressive schedule:
  #0 stabilize res=256 images=50000 batch=32 zip=data/train_50k_256.zip
  #1 fade res=512 images=400000 batch=16 zip=data/train_50k_512.zip
  #2 stabilize res=512 images=600000 batch=16 zip=data/train_50k_512.zip
  #3 fade res=1024 images=200000 batch=4 zip=data/train_50k_1024.zip
  #4 stabilize res=1024 images=500000 batch=4 zip=data/train_50k_1024.zip
Checkpoint backup dir: /content/drive/MyDrive/OpenAILab/project2/pggan_1024_fp32_safer
Initializing from checkpoint: ckpt/ffhq256_baseline.pt
  G: loaded 65 tensors with progressive remap (source_res=256), skipped 0
  G: loaded 65 tensors with progressive remap (source_res=256), skipped 0
  D: loaded 97 tensors with progressive remap (source_res=256), skipped 0
wandb: [wandb.login()]

: 

In [ ]:
# 9. Generate Samples from Progressive Checkpoint
!python generate.py --ckpt /content/drive/MyDrive/OpenAILab/project2/pggan_1024_fp32_stable/ckpt_000500176.pt --out progressive_samples.png --n 16 --nrow 8 --seed 12345


In [ ]:
# 10. Export Progressive Generator to ONNX
!python export_onnx.py --ckpt /content/drive/MyDrive/OpenAILab/project2/pggan_1024/final.pt --out submission.onnx
